# CE vs Weighted CE vs Focal Loss — Section 4

**Cross-Entropy (CE):**
$$\mathcal{L}_{\text{CE}}(p, y) = -[y \log p + (1-y)\log(1-p)]$$

**Weighted Cross-Entropy (WCE):** assigns class-specific weight $w_y$
$$\mathcal{L}_{\text{WCE}}(p, y) = -[w_1 y \log p + w_0 (1-y)\log(1-p)]$$

**Focal Loss** (Lin et al. 2017): down-weights easy examples via $(1-p_t)^\gamma$
$$\mathcal{L}_{\text{FL}}(p, y) = -\alpha_t (1-p_t)^\gamma \log(p_t)$$
where $p_t = p$ if $y=1$ else $1-p$, and $\alpha_t = \alpha$ if $y=1$ else $1-\alpha$.
Default: $\gamma=2$, $\alpha=0.75$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

eps = 1e-7

def ce_y1(p): return -np.log(np.clip(p, eps, 1-eps))
def ce_y0(p): return -np.log(np.clip(1-p, eps, 1-eps))
def wce_y1(p, w1): return w1 * ce_y1(p)
def wce_y0(p, w0): return w0 * ce_y0(p)
def focal_y1(p, gamma, alpha):
    pt = np.clip(p, eps, 1-eps)
    return -alpha * (1 - pt)**gamma * np.log(pt)
def focal_y0(p, gamma, alpha):
    pt = np.clip(1-p, eps, 1-eps)
    return -(1-alpha) * (1 - pt)**gamma * np.log(pt)


def draw_loss(w1=3.0, w0=1.0, gamma=2.0, alpha=0.75, p_val=0.3,
              show_ce=True, show_wce=True, show_fl=True):
    ps = np.linspace(0.01, 0.99, 300)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = {'CE': '#2563eb', 'WCE': '#dc2626', 'FL': '#059669'}

    for ax, y, title in zip(axes, [1, 0], ['Minority class (y=1)', 'Majority class (y=0)']):
        if show_ce:
            loss = ce_y1(ps) if y == 1 else ce_y0(ps)
            ax.plot(ps, loss, color=colors['CE'], lw=2, label='CE')
        if show_wce:
            loss = wce_y1(ps, w1) if y == 1 else wce_y0(ps, w0)
            ax.plot(ps, loss, color=colors['WCE'], lw=2, ls='--',
                    label=f'WCE (w={w1 if y==1 else w0:.1f})')
        if show_fl:
            loss = focal_y1(ps, gamma, alpha) if y == 1 else focal_y0(ps, gamma, alpha)
            ax.plot(ps, loss, color=colors['FL'], lw=2, ls=':',
                    label=f'Focal (γ={gamma:.1f}, α={alpha:.2f})')

        ax.axvline(p_val, color='#d97706', lw=2, ls=':', label=f'p={p_val:.2f}')
        ax.set_xlim(0, 1); ax.set_ylim(0, 4)
        ax.set_xlabel('Model output p'); ax.set_ylabel('Loss')
        ax.set_title(title, fontsize=11)
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    # Print values at p_val
    print(f'At p = {p_val:.2f}:')
    print(f'  CE:    y=1 → {ce_y1(p_val):.4f}   y=0 → {ce_y0(p_val):.4f}')
    print(f'  WCE:   y=1 → {wce_y1(p_val,w1):.4f}   y=0 → {wce_y0(p_val,w0):.4f}')
    print(f'  Focal: y=1 → {focal_y1(p_val,gamma,alpha):.4f}   y=0 → {focal_y0(p_val,gamma,alpha):.4f}')
    mod = (1 - p_val)**gamma
    print(f'  Focal modulator (1-p)^γ = {mod:.4f}  '
          f'({"easy—suppressed" if mod<0.1 else "hard—amplified" if mod>0.5 else "moderate"})')

    plt.suptitle(f'CE vs Weighted CE vs Focal Loss   w₁={w1:.1f}, γ={gamma:.1f}, α={alpha:.2f}',
                 fontsize=10, y=0.0)
    plt.tight_layout()
    plt.show()


widgets.interact(
    draw_loss,
    w1=widgets.FloatSlider(min=0.5, max=10, step=0.5, value=3.0, description='w₁ (class 1)', continuous_update=False),
    w0=widgets.FloatSlider(min=0.1, max=3, step=0.1, value=1.0, description='w₀ (class 0)', continuous_update=False),
    gamma=widgets.FloatSlider(min=0, max=5, step=0.25, value=2.0, description='γ (focal)', continuous_update=False),
    alpha=widgets.FloatSlider(min=0.05, max=0.95, step=0.05, value=0.75, description='α (focal)', continuous_update=False),
    p_val=widgets.FloatSlider(min=0.01, max=0.99, step=0.01, value=0.3, description='p (at cursor)', continuous_update=False),
    show_ce=widgets.Checkbox(value=True, description='Show CE'),
    show_wce=widgets.Checkbox(value=True, description='Show WCE'),
    show_fl=widgets.Checkbox(value=True, description='Show Focal'),
)